<a href="https://colab.research.google.com/github/Kingtheblaze/task/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kingtheblaze/task/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
What One Row Means (Grain): One row represents one unique pseudonymized content item (content_hash_id) aggregated over the historical feature window of March 2026 (2026-03-01 to 2026-03-31).

Which Table(s) Used: dim_content.parquet (content metadata) joined with fact_content_daily_performance (daily search and analytics facts).

Which Time Window:

Feature Window: 2026-03-01 to 2026-03-31 (1 month of historical observations).

Target Window (for future outcome): 2026-04-01 to 2026-04-30 (used strictly for creating ground-truth labels).

What to Predict / Rank: An Engagement Risk & CTR Opportunity Score that ranks visible content items (high impressions) that suffer from below-expected user engagement or CTR relative to their position tier.

One Thing Deliberately Excluded: Product decision flags (health_score, priority_score, action_type), raw unscrambled URLs/queries, and any performance metrics measured after 2026-03-31 from the feature matrix.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

# -------------------------------------------------------------------------
# SETUP & AUTHENTICATION
# -------------------------------------------------------------------------
HF_TOKEN = "YOUR_ACTUAL_TOKEN_HERE"  # Replace with your 'hf_' token

print("Loading dataset splits from Hugging Face...")
facts_ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    token=HF_TOKEN,
)

dim_ds = load_dataset(
    "FlyRank/internship-warehouse",
    data_files="dim_content.parquet",
    split="train",
    token=HF_TOKEN,
)

con = duckdb.connect()

# =========================================================================
# TASK TWO: THREE PROOF QUERIES (MID-PANEL MONTH: 2026-03)
# =========================================================================
print("\n--- TASK TWO: DATA PROOF QUERIES ---")

# Query 1: Prove Grain (Total aggregated rows vs Distinct content items)
q1_grain = """
SELECT
    COUNT(*) AS total_aggregated_rows,
    COUNT(DISTINCT content_hash_id) AS distinct_content_items
FROM (
    SELECT content_hash_id
    FROM facts_ds
    WHERE CAST(report_date AS VARCHAR) BETWEEN '2026-03-01' AND '2026-03-31'
    GROUP BY content_hash_id
)
"""
df_grain = con.sql(q1_grain).df()
print("\n[Query 1: Grain Check]")
print(df_grain)

# Query 2: Row Count and Date Span of March 2026 Slice
q2_slice = """
SELECT
    COUNT(*) AS total_fact_rows,
    MIN(CAST(report_date AS VARCHAR)) AS min_date,
    MAX(CAST(report_date AS VARCHAR)) AS max_date
FROM facts_ds
WHERE CAST(report_date AS VARCHAR) BETWEEN '2026-03-01' AND '2026-03-31'
"""
df_slice = con.sql(q2_slice).df()
print("\n[Query 2: Slice Row Count & Date Span]")
print(df_slice)

# Query 3: Availability Filter with 'IS TRUE'
q3_availability = """
SELECT
    COUNT(*) AS total_march_fact_rows,
    COUNT(CASE WHEN ga4_data_available IS TRUE THEN 1 END) AS surviving_ga4_rows,
    ROUND(COUNT(CASE WHEN ga4_data_available IS TRUE THEN 1 END) * 100.0 / COUNT(*), 2) AS survival_pct
FROM facts_ds
WHERE CAST(report_date AS VARCHAR) BETWEEN '2026-03-01' AND '2026-03-31'
"""
df_avail = con.sql(q3_availability).df()
print("\n[Query 3: Availability Filter (ga4_data_available IS TRUE)]")
print(df_avail)


# =========================================================================
# TASK THREE: 5-FEATURE FRAME CONSTRUCTION
# =========================================================================
print("\n--- TASK THREE: FEATURE FRAME GENERATION ---")

feature_query = """
WITH march_features AS (
    SELECT
        content_hash_id,
        SUM(impressions) AS total_impressions,
        SUM(clicks) AS total_clicks,
        SUM(sessions) AS total_sessions,
        AVG(position) AS avg_position,
        AVG(engagement_rate) AS avg_engagement_rate
    FROM facts_ds
    WHERE CAST(report_date AS VARCHAR) BETWEEN '2026-03-01' AND '2026-03-31'
      AND ga4_data_available IS TRUE
    GROUP BY content_hash_id
    HAVING SUM(impressions) >= 100
),
april_targets AS (
    -- Next month's performance used ONLY to generate the ground-truth label
    SELECT
        content_hash_id,
        AVG(engagement_rate) AS next_month_engagement
    FROM facts_ds
    WHERE CAST(report_date AS VARCHAR) BETWEEN '2026-04-01' AND '2026-04-30'
      AND ga4_data_available IS TRUE
    GROUP BY content_hash_id
)
SELECT
    c.content_hash_id,
    -- Feature 1
    LN(f.total_impressions + 1) AS log_impressions,
    -- Feature 2
    f.avg_position,
    -- Feature 3
    CASE WHEN f.total_impressions > 0 THEN (f.total_clicks * 1.0 / f.total_impressions) ELSE 0 END AS ctr,
    -- Feature 4
    c.word_count,
    -- Feature 5
    c.content_age_days,

    -- Ground Truth Target Label (Future outcome: low engagement next month)
    CASE WHEN t.next_month_engagement < 0.35 THEN 1 ELSE 0 END AS target_engagement_risk,

    -- LEAKAGE COLUMN (Created on purpose for Task Four)
    -- Directly encodes future engagement from April 2026!
    (t.next_month_engagement + 0.001) AS leaked_future_engagement_signal

FROM dim_ds c
JOIN march_features f ON c.content_hash_id = f.content_hash_id
JOIN april_targets t ON c.content_hash_id = t.content_hash_id
WHERE c.word_count IS NOT NULL
"""

df_features = con.sql(feature_query).df().dropna()
print(f"Feature frame built successfully. Shape: {df_features.shape}")

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Feature NameDescriptionKnowable Justificationlog_impressionsLog-transformed 30-day GSC impressionsKnowable at decision moment because it aggregates historical search exposure measured up to March 31, 2026.avg_positionAverage SERP ranking positionKnowable at decision moment because it reflects past page search visibility within March 2026.ctrHistorical Click-Through RateKnowable at decision moment because it is derived strictly from historical clicks and impressions observed during March 2026.word_countLength of the content in wordsKnowable at decision moment because it is a static document attribute extracted directly from content metadata.content_age_daysContent age in days from publicationKnowable at decision moment because it is calculated relative to the March 31, 2026 decision point.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# =========================================================================
# TASK FOUR: THE LEAKAGE TRAP DEMONSTRATION
# =========================================================================
print("\n--- TASK FOUR: THE LEAKAGE TRAP ---")

feature_cols = [
    "log_impressions",
    "avg_position",
    "ctr",
    "word_count",
    "content_age_days",
]
target_col = "target_engagement_risk"
leak_col = "leaked_future_engagement_signal"

# Split data into train and test sets
X = df_features[feature_cols + [leak_col]]
y = df_features[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -------------------------------------------------------------------------
# STEP 1: Train WITH the Leaked Column
# -------------------------------------------------------------------------
model_leaked = RandomForestClassifier(
    n_estimators=100, max_depth=5, random_state=42
)
model_leaked.fit(X_train[feature_cols + [leak_col]], y_train)

probs_leaked = model_leaked.predict_proba(X_test[feature_cols + [leak_col]])[
    :, 1
]
auc_leaked = roc_auc_score(y_test, probs_leaked)

print(f"\n🚨 [WITH LEAKAGE TRAP] ROC-AUC Score: {auc_leaked:.4f}")
print(
    "Notice how the score approaches 1.000 (perfect). The model isn't smart—it cheated by using future data!"
)

# -------------------------------------------------------------------------
# STEP 2: Drop the Leaked Column & Train Honest Model
# -------------------------------------------------------------------------
model_honest = RandomForestClassifier(
    n_estimators=100, max_depth=5, random_state=42
)
model_honest.fit(X_train[feature_cols], y_train)

probs_honest = model_honest.predict_proba(X_test[feature_cols])[:, 1]
auc_honest = roc_auc_score(y_test, probs_honest)

print(f"\n✅ [HONEST MODEL] ROC-AUC Score: {auc_honest:.4f}")
print(
    "The honest score drops back to reality. This is the true baseline performance using observable signals only."
)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.